# Notebook 08: MLP training on bike rentals by hand

In this notebook, we will train a small multilayer perceptron (MLP) to predict bike-rental counts from a curated real dataset.

The goal is not to use a machine-learning library. The goal is to see the whole training path with plain Python lists and floats: data row -> prediction -> loss -> gradients -> parameter update.

We will start by loading one row, naming the five model inputs, and comparing the scaled target with the original rental count. Then we will build the forward pass, derive the backward pass by hand, and train with plain gradient descent.


## First example: one row becomes model inputs and a target

The CSV file has one bike-rental example per row. A **model input** is a number the model is allowed to use for its prediction. A **target** is the answer the model is trying to predict.

For the first row, we will use these five inputs: `hour_scaled`, `temperature`, `humidity`, `wind_speed`, and `working_day`. The scaled target is `target_scaled`, which is the rental count divided by `1000`.


In [1]:
import csv

from pathlib import Path

DATA_PATH = "../data/bike_rentals_mini.csv"

rows = []

with Path(DATA_PATH).open(newline="") as csv_file:
    reader = csv.DictReader(csv_file)
    column_names = reader.fieldnames

    for row in reader:
        rows.append(row)

first_row = rows[0]

input_values = [
    float(first_row["hour_scaled"]),
    float(first_row["temperature"]),
    float(first_row["humidity"]),
    float(first_row["wind_speed"]),
    float(first_row["working_day"]),
]

target_scaled = float(first_row["target_scaled"])

print("columns:", column_names)
print("number of rows:", len(rows))
print("inputs:", input_values)
print("target_scaled:", target_scaled)
print("original rental_count:", first_row["rental_count"])

columns: ['hour', 'hour_scaled', 'temperature', 'humidity', 'wind_speed', 'working_day', 'rental_count', 'target_scaled']
number of rows: 240
inputs: [0.6956521739130435, 0.3, 0.49, 0.2537, 1.0]
target_scaled: 0.083
original rental_count: 83


## Model shape

Each training example gives the model five input numbers:

1. `hour_scaled`
2. `temperature`
3. `humidity`
4. `wind_speed`
5. `working_day`

The model will use this shape:

```text
5 inputs -> 6 ReLU hidden neurons -> 1 output
```

Each hidden neuron produces one hidden value. The output layer combines those hidden values into one scaled rental-count prediction.


## Forward pass as a reusable function

A forward pass sends one example through the model to produce one prediction. We will put the forward pass inside a function right away, because training needs to reuse the same calculation for many rows.

Inside the function, Layer 1 turns the five input values into hidden ReLU values. ReLU means “keep positive numbers, turn negative numbers into `0`.”

Layer 2 combines the hidden values into one scaled rental-count prediction. The function also returns a cache, which stores intermediate values the backward pass will need when it computes gradients.


In [2]:
def relu(value: float) -> float:
    return max(0.0, value)


ForwardCache = dict[str, list[float]]

hidden_size = 6

# Parameters for the input -> hidden layer
input_to_hidden_weights = [
    [0.10, -0.20, 0.05, 0.03, 0.08],
    [-0.04, 0.12, -0.07, 0.09, 0.02],
    [0.06, 0.01, 0.11, -0.05, -0.03],
    [-0.08, 0.04, 0.02, 0.10, 0.07],
    [0.03, -0.06, 0.09, 0.01, -0.04],
    [0.05, 0.07, -0.02, -0.08, 0.06],
]
hidden_biases = [0.01, -0.02, 0.00, 0.03, -0.01, 0.02]

# Parameters for the hidden -> output layer
hidden_to_output_weights = [0.20, -0.10, 0.15, 0.05, -0.08, 0.12]
output_bias = 0.01


def forward_pass(
    input_values: list[float],
    input_to_hidden_weights: list[list[float]],
    hidden_biases: list[float],
    hidden_to_output_weights: list[float],
    output_bias: float,
) -> tuple[float, ForwardCache]:
    hidden_raw_values = []
    hidden_outputs = []

    # Layer 1: input values -> hidden ReLU values
    for hidden_index in range(len(hidden_biases)):
        raw_value = hidden_biases[hidden_index]

        for input_index in range(len(input_values)):
            raw_value += (
                input_values[input_index]
                * input_to_hidden_weights[hidden_index][input_index]
            )

        hidden_output = relu(raw_value)

        hidden_raw_values.append(raw_value)
        hidden_outputs.append(hidden_output)

    # Layer 2: hidden ReLU values -> one scaled prediction
    prediction = output_bias

    for hidden_index in range(len(hidden_outputs)):
        prediction += hidden_outputs[hidden_index] * hidden_to_output_weights[hidden_index]

    cache = {
        "input_values": input_values,
        "hidden_raw_values": hidden_raw_values,
        "hidden_outputs": hidden_outputs,
    }

    return prediction, cache


prediction, cache = forward_pass(
    input_values,
    input_to_hidden_weights,
    hidden_biases,
    hidden_to_output_weights,
    output_bias,
)

print("prediction_scaled:", prediction)
print("target_scaled:", target_scaled)
print("cached hidden raw values:", cache["hidden_raw_values"])
print("cached hidden outputs:", cache["hidden_outputs"])


prediction_scaled: 0.06198664739130435
target_scaled: 0.083
cached hidden raw values: [0.13167621739130436, -0.003293086956521748, 0.055954130434782604, 0.09151782608695652, -0.000493434782608701, 0.10568660869565218]
cached hidden outputs: [0.13167621739130436, 0.0, 0.055954130434782604, 0.09151782608695652, 0.0, 0.10568660869565218]
